# Prediction for Picnic Weather
## Apply Model for Prediction

Apply fitted model with Weather data from API request

Use stored XGB model

---

## 1. Load Modules & Load Data



In [23]:

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from core.data import load_from_kaggle

# Styling für bessere Visualisierungen
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# GeoPy with Nominatim to estimate lat, lon from city name
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import time
import requests     # um die OpenAPI für Höhenangaben bei lat, lon abzufragen

# Meteostat for weather data
from datetime import date
import meteostat as ms

# Models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib   # um Modelle zu speichern und zu laden
# from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance

# import module for Score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import cross_val_score



## 2. Load Model from File



In [24]:
# Load model from file
def load_model(model_name):
    model_path = f'{model_name}.joblib'
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        # print(f"✅ Modell '{model_name}' erfolgreich geladen.")
        return model
    else:
        print(f"❌ Modell '{model_name}' nicht gefunden unter: {model_path}")
        return None


In [25]:

file_weather_FS    = '../data/processed/weather_FS_lat_lon_elevation.csv'
file_weather_meteostat    = '../data/processed/weather_meteostat_lat_lon_elevation.csv'

# For extended analysis, you could also load the complete dataset if needed
weather_FS = pd.read_csv(file_weather_FS, 
                           delimiter=',', encoding='ascii', parse_dates=['DATE'])
print(f"📊 Geladene Datei: {file_weather_FS}")
print(f"📏 Shape: {weather_FS.shape[0]:,} Zeilen × {weather_FS.shape[1]} Spalten\n")

# For extended analysis, you could also load the complete dataset if needed
meteostat = pd.read_csv(file_weather_meteostat, 
                           delimiter=',', encoding='ascii', parse_dates=['DATE'])
print(f"📊 Geladene Datei: {file_weather_meteostat}")
print(f"📏 Shape: {meteostat.shape[0]:,} Zeilen × {meteostat.shape[1]} Spalten\n")

# Convert the DATE column to datetime. The DATE is given as an integer (likely in YYYYMMDD format)
# for df in [weather_FS, meteostat]:
#     df['DATE'] = pd.to_datetime(df['DATE'].astype(str), format='%Y%m%d', errors='coerce')

print('Data loaded and DATE column converted to datetime.')

display(weather_FS.sample(3))
meteostat.sample(3)

weather_FS.info()
meteostat.info()

📊 Geladene Datei: ../data/processed/weather_FS_lat_lon_elevation.csv
📏 Shape: 62,118 Zeilen × 18 Spalten

📊 Geladene Datei: ../data/processed/weather_meteostat_lat_lon_elevation.csv
📏 Shape: 58,379 Zeilen × 18 Spalten

Data loaded and DATE column converted to datetime.


,DATE,MONTH,cloud_cover,wind_speed,wind_gust,humidity,pressure,global_radiation,precipitation,sunshine,temp_mean,temp_min,temp_max,city,picnic_weather,lat,lon,elevation
53095,2005-04-23,4,4.0,NaN,NaN,0.63,NaN,3.23,0.00,12.7,-7.2,-10.6,-3.7,SONNBLICK,False,46.900346,13.383461,2427.0
8431,2003-01-28,1,7.0,6.2,16.0,0.81,1.0089,0.20,0.49,0.3,6.1,4.0,8.6,DE_BILT,False,52.144559,5.173777,3.0
17045,2006-08-26,8,6.0,2.1,7.8,0.80,1.0094,1.40,0.01,3.7,16.0,12.1,20.0,DUSSELDORF,False,51.225402,6.776314,44.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62118 entries, 0 to 62117
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   DATE              62118 non-null  datetime64[ns]
 1   MONTH             62118 non-null  int64         
 2   cloud_cover       43848 non-null  float64       
 3   wind_speed        43848 non-null  float64       
 4   wind_gust         25578 non-null  float64       
 5   humidity          54810 non-null  float64       
 6   pressure          51156 non-null  float64       
 7   global_radiation  54810 non-null  float64       
 8   precipitation     62118 non-null  float64       
 9   sunshine          47502 non-null  float64       
 10  temp_mean         62118 non-null  float64       
 11  temp_min          58464 non-null  float64       
 12  temp_max          62118 non-null  float64       
 13  city              62118 non-null  object        
 14  picnic_weather    6211



---

## 3. Request Weather Data from Meteostat API



In [58]:
# Function to request Weather Data from Meteostat API for a cityname and a date(one day) and 
# return lat, lon, elevation, and daily weather values
from datetime import datetime


def get_weather_data_meteostat(city_name, date_str):
    geolocator = Nominatim(user_agent="weather_app")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    location = geocode(city_name)
    if location is None:
        print(f"❌ Fehler: Stadt '{city_name}' nicht gefunden.")
        return None
    
    lat, lon = location.latitude, location.longitude
    # print(f"📍 Geokodierung für '{city_name}': lat={lat}, lon={lon}")
    
    # Open-Meteo API für Höhenangaben
    elevation_url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        response = requests.get(elevation_url)
        response.raise_for_status()
        elevation_data = response.json()
        elevation = elevation_data.get('elevation', None)
        # print(f"⛰️ Höhenangabe für '{city_name}': {elevation} m")
    except requests.RequestException as e:
        print(f"❌ Fehler bei der Höhenabfrage: {e}")
        elevation = None
    
    # Meteostat API für Wetterdaten
    start_date = end_date = datetime.strptime(date_str, '%Y-%m-%d') # date(date_str)
    try:
        # data = ms.daily(ms.Point(lat, lon), start_date, end_date)
        # data = data.fetch()
        # Specify location and time range
        POINT = ms.Point(lat, lon, elevation)
        # START = date(2000, 1, 1)
        # END = date(2010, 1, 1)
        # Get nearby weather stations
        stations = ms.stations.nearby(POINT, limit=4)

        # Get daily data & perform interpolation
        ts = ms.daily(stations, start_date, end_date)
        data = ms.interpolate(ts, POINT).fetch()

        # Reset index so the time index becomes a normal column and can be renamed
        data = data.reset_index()

        # Rename columns to match weather_FS naming
        data = data.rename(columns={
            'time': 'DATE',
            'temp': 'temp_mean',
            'tmin': 'temp_min',
            'tmax': 'temp_max',
            'rhum': 'humidity',
            'prcp': 'precipitation',
            'snwd': 'snowfall',
            'wspd': 'wind_speed',
            'pres': 'pressure',
            'tsun': 'sunshine',
            'cldc': 'cloud_cover',
            'wpgt': 'wind_gust'
        })

        # Convert sunshine from minutes to hours
        if 'sunshine' in data.columns:
            data['sunshine'] = data['sunshine'] / 60
        if 'wind_gust' in data.columns:
            data['wind_gust'] = data['wind_gust']
        else:
            data['wind_gust'] = data['wind_speed'] * 1.5  # Beispielhafter Ansatz, anpassen nach Bedarf
        
        data['MONTH'] = data['DATE'].dt.month
        # data['wind_gust'] = data['wind_speed'] * 1.5  # Beispielhafter Ansatz, anpassen nach Bedarf
        data['global_radiation'] = 0.18 * data['sunshine'] + 0.4  # Beispielhafter Ansatz, anpassen nach Bedarf
        data['lat'] = lat
        data['lon'] = lon
        data['elevation'] = elevation
        

        if data.empty:
            print(f"⚠️ Keine Wetterdaten für '{city_name}' am {date_str} gefunden.")
            return None
        # else:
            # print(f"✅ Wetterdaten für '{city_name}' am {date_str} gefunden: {data.columns.tolist()}")
            # display(data)
        



        weather_info = {
            'city': city_name,
            'date': date_str,
            'lat': lat,
            'lon': lon,
            'elevation': elevation,
            'temperature': data['temp_mean'].iloc[0] if 'temp_mean' in data.columns else None,
            'precipitation': data['precipitation'].iloc[0] if 'precipitation' in data.columns else None,
            'wind_speed': data['wind_speed'].iloc[0] if 'wind_speed' in data.columns else None,
            # Weitere relevante Wetterparameter können hier hinzugefügt werden
        }
        
        # print(f"✅ Wetterdaten für '{city_name}' am {date_str} erfolgreich abgerufen.")
        return data # weather_info
    except Exception as e:
        print(f"❌ Fehler bei der Wetterabfrage: {e}")
        return None


In [27]:
# Function to request Weather Data from Open-Meteo API for a cityname and a date(one day) and 
# return lat, lon, elevation, and daily weather values

# pip install openmeteo-requests
# pip install requests-cache retry-requests numpy pandas

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

def get_weather_data_openmeteo(city_name, date_str):
    geolocator = Nominatim(user_agent="weather_app")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    location = geocode(city_name)
    if location is None:
        print(f"❌ Fehler: Stadt '{city_name}' nicht gefunden.")
        return None
    
    lat, lon = location.latitude, location.longitude
    # print(f"📍 Geokodierung für '{city_name}': lat={lat}, lon={lon}")
    
    # Open-Meteo API für Höhenangaben
    elevation_url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        response = requests.get(elevation_url)
        response.raise_for_status()
        elevation_data = response.json()
        elevation = elevation_data.get('elevation', None)
        # print(f"⛰️ Höhenangabe für '{city_name}': {elevation} m")
    except requests.RequestException as e:
        print(f"❌ Fehler bei der Höhenabfrage: {e}")
        elevation = None
    
    # Meteostat API für Wetterdaten
    # start_date = end_date = datetime.strptime(date_str, '%Y-%m-%d') # date(date_str)

    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    # Make sure all required weather variables are listed here
    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        # "daily": ["temperature_2m_max", "temperature_2m_min", "sunshine_duration", "precipitation_sum", "precipitation_probability_max", "temperature_2m_mean", "relative_humidity_2m_mean", "snowfall_sum", "wind_gusts_10m_mean", "wind_speed_10m_mean", "surface_pressure_mean", "cloud_cover_mean"],
        "daily": ["temperature_2m_max", "temperature_2m_min", "sunshine_duration", "precipitation_sum", 
                  "precipitation_probability_max", "temperature_2m_mean", "relative_humidity_2m_mean", 
                  "snowfall_sum", "wind_gusts_10m_mean", "wind_speed_10m_mean", "surface_pressure_mean", 
                  "cloud_cover_mean", "shortwave_radiation_sum", "precipitation_hours"],
        "start_date": date_str,
        "end_date": date_str,
        # "forecast_days": 16,
    }
    responses = openmeteo.weather_api(url, params = params)

    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    # print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    # print(f"Elevation: {response.Elevation()} m asl")
    # print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

    # Process daily data. The order of variables needs to be the same as requested.
    daily = response.Daily()
    daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
    daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
    daily_sunshine_duration = daily.Variables(2).ValuesAsNumpy()
    daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
    daily_precipitation_probability_max = daily.Variables(4).ValuesAsNumpy()
    daily_temperature_2m_mean = daily.Variables(5).ValuesAsNumpy()
    daily_relative_humidity_2m_mean = daily.Variables(6).ValuesAsNumpy()
    daily_snowfall_sum = daily.Variables(7).ValuesAsNumpy()
    daily_wind_gusts_10m_mean = daily.Variables(8).ValuesAsNumpy()
    daily_wind_speed_10m_mean = daily.Variables(9).ValuesAsNumpy()
    daily_surface_pressure_mean = daily.Variables(10).ValuesAsNumpy()
    daily_cloud_cover_mean = daily.Variables(11).ValuesAsNumpy()
    daily_shortwave_radiation_sum = daily.Variables(12).ValuesAsNumpy()
    daily_precipitation_hours = daily.Variables(13).ValuesAsNumpy()

    daily_data = {
        "date": pd.date_range(
            start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = daily.Interval()),
            inclusive = "left"
        )
    }

    daily_data["DATE"] = datetime.strptime(date_str, '%Y-%m-%d')
    daily_data["MONTH"] = datetime.strptime(date_str, '%Y-%m-%d').month
    daily_data["temp_mean"] = daily_temperature_2m_mean
    daily_data["temp_max"] = daily_temperature_2m_max
    daily_data["temp_min"] = daily_temperature_2m_min
    daily_data["cloud_cover"] = daily_cloud_cover_mean
    daily_data["wind_speed"] = daily_wind_speed_10m_mean
    daily_data["sunshine"] = daily_sunshine_duration / 3600  # Convert seconds to hours
    daily_data["wind_gust"] = daily_wind_gusts_10m_mean
    daily_data["humidity"] = daily_relative_humidity_2m_mean
    daily_data["pressure"] = daily_surface_pressure_mean
    daily_data["global_radiation"] = daily_shortwave_radiation_sum
    daily_data["lat"] = lat
    daily_data["lon"] = lon
    daily_data["elevation"] = elevation
    daily_data["precipitation"] = daily_precipitation_sum
    daily_data["precipitation_probability_max"] = daily_precipitation_probability_max
    daily_data["snowfall"] = daily_snowfall_sum
    daily_data["precipitation_hours"] = daily_precipitation_hours

    daily_dataframe = pd.DataFrame(data = daily_data)
    # print("\nDaily data\n", daily_dataframe)
    # display(daily_dataframe)
    
    return daily_dataframe


In [59]:

df_picnic = get_weather_data_meteostat('Frankfurt', '2025-08-14')
# df_picnic = get_weather_data_openmeteo('Frankfurt', '2026-05-07')
df_picnic

,DATE,temp_mean,temp_min,temp_max,humidity,precipitation,snowfall,wind_speed,wind_gust,pressure,sunshine,cloud_cover,MONTH,global_radiation,lat,lon,elevation
0,2025-08-14,28.0,20.2,36.2,54,0.0,0,6.4,24.1,1017.2,12.083333,4,8,2.575,50.110644,8.682092,103.0


In [29]:
europe_features = {
    'EUROall': ['cloud_cover', 'wind_speed', 'wind_gust', 'humidity', 'pressure', 'global_radiation',
              'precipitation', 'sunshine', 'temp_mean', 'temp_min', 'temp_max', 'lat', 'lon', 'elevation'],
    # 'EUROselect': ['MONTH', 'precipitation', 'temp_mean', 'temp_max', 'lat', 'lon', 'elevation'],
    'EUROselect': ['MONTH', 'temp_mean', 'temp_max', 'cloud_cover', 'wind_speed', 'sunshine', 
                   'wind_gust', 'humidity', 'pressure', 'global_radiation', 'lat', 'lon', 'elevation'],
    'EUROmin': ['MONTH', 'temp_mean', 'temp_max', 'lat', 'lon', 'elevation'],
              }

In [30]:
def picnic_prediction(city_name, date_str):
    df_picnic = get_weather_data_openmeteo(city_name, date_str)
    
    model = load_model('model_xgb_EURO')
    
    picnic = model.predict(pd.DataFrame(df_picnic[europe_features['EUROselect']]))
    
    if picnic[0] == 1:
        print(f"✅ Das Wetter in {city_name} am {date_str} ist ideal für ein Picknick! 🌞🍉")
    else:
        print(f"❌ Das Wetter in {city_name} am {date_str} ist nicht ideal für ein Picknick. 🌧️☔")
    # return picnic

In [68]:
def picnic_prediction_past(city_name, date_str):
    df_picnic = get_weather_data_meteostat(city_name, date_str)
    
    model = load_model('model_xgb_EURO')
    
    picnic = model.predict(pd.DataFrame(df_picnic[europe_features['EUROselect']]))
    
    if picnic[0] == 1:
        print(f"✅ Das Wetter in {city_name} am {date_str} war ideal für ein Picknick! 🌞🍉")
    else:
        print(f"❌ Das Wetter in {city_name} am {date_str} war nicht ideal für ein Picknick. 🌧️☔")
    return picnic[0]

In [63]:
picnic_prediction_past('Basel', '2013-08-05')

✅ Das Wetter in Basel am 2013-08-05 war ideal für ein Picknick! 🌞🍉


In [64]:
picnic_prediction('Madrid', '2026-05-07')

❌ Das Wetter in Madrid am 2026-05-07 ist nicht ideal für ein Picknick. 🌧️☔


In [69]:
def picnic_prediction2(city_name, date_str):
    df_picnic = get_weather_data_openmeteo(city_name, date_str)
    
    model = load_model('model_xgb_EURO')
    
    picnic = model.predict(pd.DataFrame(df_picnic[europe_features['EUROselect']]))
    
    if picnic[0] == 1:
        print(f"✅ Das Wetter in {city_name} am {date_str} ist ideal für ein Picknick! 🌞🍉")
    else:
        print(f"❌ Das Wetter in {city_name} am {date_str} ist nicht ideal für ein Picknick. 🌧️☔")
    return picnic[0]



---

## 7. Interaktive Display



In [33]:
# Installiere die benötigten Bibliotheken (falls nicht vorhanden)
# !pip install ipywidgets

from ipywidgets import interact, widgets, Output
from IPython.display import display, clear_output
from datetime import datetime, date, timedelta  # Korrekter Import

In [65]:
# Funktion zum Anzeigen des Bildes basierend auf dem Rückgabewert
def show_result(city, date):
    with output:
        clear_output()
        date_str = date_picker.value.strftime("%Y-%m-%d")
        if date >= datetime.now().date():
            print(f"🔮 Vorhersage für {city} am {date_str}:")
            result = picnic_prediction2(city, date_str)
        else:
            print(f"📜 Rückblick für {city} am {date_str}:")
            result = picnic_prediction_past(city, date_str)
        if result == 1:
            # display(widgets.HTML('<img src="bild1.jpg" width="400">'))
            display(widgets.HTML('<img src="https://images.stockcake.com/public/6/c/5/6c59f6ba-39d4-4f98-b6d2-ff786a0c499d_large/sunlit-picnic-fruits-stockcake.jpg" width="400">'))
        else:
            # display(widgets.HTML('<img src="bild0.jpg" width="400">'))
            display(widgets.HTML('<img src="https://thumbs.dreamstime.com/b/weather-symbol-hot-rain-simple-81380664.jpg" width="400">'))
        

In [67]:
# Widgets erstellen
city_input = widgets.Text(
    description='Stadt:',
    placeholder='Berlin',
    disabled=False
)

# Datumsauswahl: Heute bis heute + 16 Tage
today = date.today()
date_picker = widgets.DatePicker(
    description='Datum:',
    disabled=False,
    value=today,
    min=date(2000, 1, 1),  # Optional: Setze ein Mindestdatum
    max=today + timedelta(days=16)
)

# Button für die Vorhersage
button = widgets.Button(
    description='Vorhersage',
    disabled=False,
    button_style='',  # 'success', 'info', 'warning', 'danger' oder ''
    tooltip='Klicken, um die Vorhersage auszuführen'
)

# Output-Bereich für das Bild
output = widgets.Output()

# Funktion für den Button-Klick
def on_button_click(b):
    print(date_picker.value)
    show_result(city_input.value, date_picker.value)

button.on_click(on_button_click)

# Anzeigen der Widgets
display(city_input, date_picker, button, output)

Text(value='', description='Stadt:', placeholder='Berlin')

DatePicker(value=datetime.date(2026, 5, 7), description='Datum:', max=datetime.date(2026, 5, 23), min=datetime…

Button(description='Vorhersage', style=ButtonStyle(), tooltip='Klicken, um die Vorhersage auszuführen')

Output()

# Installiere die benötigten Bibliotheken (falls nicht vorhanden)
# !pip install ipywidgets

from ipywidgets import interact, widgets, Output
from IPython.display import display, clear_output
import datetime

# Platzhalter für deine Funktion (ersetze sie mit deiner echten Funktion)
def vorhersage(cityname, date):
    """
    Platzhalter-Funktion: Ersetze dies mit deiner echten Vorhersage-Funktion.
    Gibt 1 oder 0 zurück.
    """
    # Beispiel: Immer 1 zurückgeben (für Tests)
    # Ersetze dies mit deiner Logik, z. B. basierend auf Wetterdaten.
    return 1  # oder 0

# Funktion zum Anzeigen des Bildes basierend auf dem Rückgabewert
def show_result(city, date):
    with output:
        clear_output()
        result = picnic_prediction2(city, date)
        if result == 1:
            <!-- display(widgets.HTML('<img src="bild1.jpg" width="400">')) -->
            display(widgets.HTML('<img src="https://thumbs.dreamstime.com/b/weather-symbol-hot-rain-simple-81380664.jpg" width="400">'))
        else:
            <!-- display(widgets.HTML('<img src="bild0.jpg" width="400">')) -->
            display(widgets.HTML('<img src="https://images.stockcake.com/public/6/c/5/6c59f6ba-39d4-4f98-b6d2-ff786a0c499d_large/sunlit-picnic-fruits-stockcake.jpg" width="400">'))

# Widgets erstellen
city_input = widgets.Text(
    description='Stadt:',
    placeholder='Berlin',
    disabled=False
)

# Datumsauswahl: Heute bis heute + 16 Tage
today = datetime.date.today()
date_picker = widgets.DatePicker(
    description='Datum:',
    disabled=False,
    value=today,
    min=today,
    max=today + datetime.timedelta(days=16)
)

# Button für die Vorhersage
button = widgets.Button(
    description='Vorhersage',
    disabled=False,
    button_style='',  # 'success', 'info', 'warning', 'danger' oder ''
    tooltip='Klicken, um die Vorhersage auszuführen'
)

# Output-Bereich für das Bild
output = widgets.Output()

# Funktion für den Button-Klick
def on_button_click(b):
    show_result(city_input.value, date_picker.value)

button.on_click(on_button_click)

# Anzeigen der Widgets
display(city_input, date_picker, button, output)



---

## 8. Zusammenfassung & Bewertung

